# Notebook 11 -- Three-Angle Evaluation on CommonsenseQA
## SLM-to-SLM Guided Reasoning Pipeline

**Why CommonsenseQA?**

CommonsenseQA tests everyday real-world reasoning — the kind of knowledge
humans pick up from lived experience rather than formal education.
Questions require understanding concepts like causality, spatial relationships,
social norms, and object properties.

This is maximally different from math: there are no equations, no numeric
targets, no formal operations. If the guided pipeline helps here, it
strongly suggests the benefit comes from **structured verbal reasoning**
in general — not from domain-specific fine-tuning knowledge.

The guide was fine-tuned on GSM8K (math). CommonsenseQA is the
hardest possible out-of-domain test for that guide.

**Pipeline Under Test:**
- Guide  : Qwen 2.5-3B-Instruct + LoRA fine-tuned adapter (1 forward pass)
- Solver : Qwen 2.5-1.5B-Instruct × 5 majority-vote passes
- Baseline: Qwen 2.5-1.5B-Instruct × 5 majority-vote passes (no guide plan)
- Random chance: **20.0%** (5 options: A–E)

**Note on dataset split:**
The CommonsenseQA official test labels are withheld for the public leaderboard.
We use the **validation split** (1,221 questions with ground-truth labels),
which is standard practice for offline evaluation.

**Three Evaluation Angles:**
1. Compute Efficiency  -- accuracy per billion parameter-passes
2. Vote Consistency    -- how reliably the ensemble agrees on the correct answer
3. Confidence Calibration -- how well confidence predicts correctness (ECE)


In [1]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q peft==0.12.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")


Done.


In [2]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login
login("")
print("HuggingFace login done")


HuggingFace login done


In [3]:
# CELL 3 -- Imports + GPU
import os, json, re, glob, random, time
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/csqa_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")


PyTorch : 2.10.0+cu128
GPU     : Tesla T4
VRAM    : 15.6 GB
Output  : /kaggle/working/csqa_eval


In [4]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "meta-llama/Llama-3.2-3B-Instruct",
    "response_model"      : "meta-llama/Llama-3.2-1B-Instruct",

    # Dataset
    "dataset_name"        : "tau/commonsense_qa",
    "dataset_split"       : "validation",  # test labels are withheld; validation has 1,221 questions
    "max_eval_samples"    : 900,
    "random_seed"         : 42,            # FIXED -- same seed for BOTH conditions

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.4,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 400,

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


Config ready:
  guide_base              : meta-llama/Llama-3.2-3B-Instruct
  response_model          : meta-llama/Llama-3.2-1B-Instruct
  dataset_name            : tau/commonsense_qa
  dataset_split           : validation
  max_eval_samples        : 900
  random_seed             : 42
  n_votes                 : 5
  vote_temperature        : 0.4
  guide_temperature       : 0.1
  refiner_temperature     : 0.3
  max_new_tokens          : 400
  guide_params_B          : 3.0
  solver_params_B         : 1.5
  results_file            : /kaggle/working/csqa_eval/results.jsonl
  report_file             : /kaggle/working/csqa_eval/eval_report.json
  angle1_file             : /kaggle/working/csqa_eval/angle1_compute_efficiency.json
  angle2_file             : /kaggle/working/csqa_eval/angle2_vote_consistency.json
  angle3_file             : /kaggle/working/csqa_eval/angle3_confidence_calibration.json
  checkpoint_file         : /kaggle/working/csqa_eval/checkpoint.json
  save_every              :

In [5]:
# CELL 5 -- Load CommonsenseQA dataset
# Fields:
#   id          : str
#   question    : str  (the question text)
#   question_concept : str (the core concept being tested)
#   choices     : dict with 'label' (list: ['A','B','C','D','E']) and 'text' (list of str)
#   answerKey   : str  (single letter A-E, only available in train/validation)
#
# We use the validation split (1,221 questions with labels).
# Format: "<question>\n\nOptions:\nA) ...\nB) ...\nC) ...\nD) ...\nE) ..."

print("Loading CommonsenseQA from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"])

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Features : {list(raw_ds[CONFIG['dataset_split']].features.keys())}")
print(f"Val size : {len(raw_ds[CONFIG['dataset_split']])}")

ex = raw_ds[CONFIG["dataset_split"]][0]
print(f"\nExample record:")
for k, v in ex.items():
    print(f"  {k}: {v}")


VALID_LETTERS = set("ABCDE")

def normalise_csqa(item):
    """Convert CommonsenseQA record to {question, answer, concept} for the pipeline."""
    labels = item["choices"]["label"]   # ['A','B','C','D','E']
    texts  = item["choices"]["text"]    # list of option strings
    options_str = "\n".join(f"{l}) {t}" for l, t in zip(labels, texts))
    q   = item["question"].strip() + "\n\nOptions:\n" + options_str
    ans = str(item["answerKey"]).strip().upper()
    return {
        "question"  : q,
        "answer"    : ans,
        "concept"   : item.get("question_concept", ""),
    }


all_data = [normalise_csqa(x) for x in raw_ds[CONFIG["dataset_split"]]]

# CRITICAL: fix seed ONCE before sampling
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

print(f"\nSample question:\n{test_data[0]['question']}")
print(f"Concept : {test_data[0]['concept']}")
print(f"Answer  : {test_data[0]['answer']}")


Loading CommonsenseQA from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9741 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1221 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1140 [00:00<?, ? examples/s]

Splits   : ['train', 'validation', 'test']
Features : ['id', 'question', 'question_concept', 'choices', 'answerKey']
Val size : 1221

Example record:
  id: 1afa02df02c908a558b4036e80242fac
  question: A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?
  question_concept: revolving door
  choices: {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['bank', 'library', 'department store', 'mall', 'new york']}
  answerKey: A

Sampled 900 questions (seed=42)

Sample question:
You can do knitting to get the feeling of what?

Options:
A) relaxation
B) arthritis
C) adrenaline
D) your
E) sweater may produced
Concept : knitting
Answer  : A


In [6]:
# CELL 6 -- Answer extraction for multiple-choice (A-E)
# CommonsenseQA answers are single letters A, B, C, D, or E.
# Extraction logic mirrors AQUA-RAT (also 5-choice).

def extract_gt_answer(answer_str):
    """GT is already a clean letter -- just uppercase and validate."""
    s = str(answer_str).strip().upper()
    return s if s in VALID_LETTERS else ""

def extract_pred_answer(text):
    """
    Extract the chosen option letter (A-E) from model free-form output.
    Priority order -- most explicit formats first.
    """
    text = text.strip()

    # 1. Conclusive answer phrases
    m = re.search(
        r"(?:the answer is|answer is|answer:|the correct answer is|correct answer is)"
        r"[\s:]*([A-E])\b",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 2. "option/choice X is correct/is the answer"
    m = re.search(
        r"(?:option|choice)\s+([A-E])\s+(?:is correct|is the answer|matches|is right|is most likely)",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 3. #### A  -- standard termination marker
    m = re.search(r"####\s*([A-E])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 4. Parenthesised at end: (A), (B) ...
    m = re.search(r"\(([A-E])\)\s*$", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 5. Bold: **A**, **A)**
    m = re.search(r"\*\*([A-E])\)?\*\*", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 6. Standalone letter on its own line (last occurrence)
    matches = re.findall(r"^\s*([A-E])\s*$", text, re.MULTILINE | re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    # 7. Last standalone letter anywhere
    matches = re.findall(r"\b([A-E])\b", text, re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    return ""

# Quick test
test_outputs = [
    "After thinking it through, the answer is C",
    "The correct answer is B.",
    "#### D",
    "(A)",
    "**E**",
]
for t in test_outputs:
    print(f"  '{t[:50]}' -> '{extract_pred_answer(t)}'")
print("Extraction OK")


  'After thinking it through, the answer is C' -> 'C'
  'The correct answer is B.' -> 'B'
  '#### D' -> 'D'
  '(A)' -> 'A'
  '**E**' -> 'E'
Extraction OK


In [7]:
# CELL 7 -- Load fine-tuned guide model (Qwen 3B + LoRA)

def find_adapter():
    patterns = [
        "/kaggle/input/datasets/fushiguro019/final-adapter-lama-svamp",
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None


print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
guide_tok.padding_side = "left"
if guide_tok.pad_token is None:
    guide_tok.pad_token = guide_tok.eos_token

guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

adapter_path = find_adapter()
if adapter_path:
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("LoRA adapter loaded -- fine-tuned guide active")
else:
    print("WARNING: No adapter found. Using base Qwen 3B as guide.")

guide_model.eval()
print(f"Guide VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading guide base: meta-llama/Llama-3.2-3B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

  Found adapter: /kaggle/input/datasets/fushiguro019/final-adapter-lama-svamp
LoRA adapter loaded -- fine-tuned guide active
Guide VRAM: 3.25 GB


In [8]:
# CELL 8 -- Load solver model (Qwen 1.5B)

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    dtype=torch.float16,
    device_map="auto",
).eval()

total_vram = torch.cuda.memory_allocated() / 1e9
headroom   = 17.1 - total_vram
print(f"Total VRAM (both models): {total_vram:.2f} GB / 17.1 GB")
print(f"Headroom                : {headroom:.1f} GB")
print("Memory OK" if headroom >= 2 else "WARNING: Tight -- reduce n_votes to 3 if OOM")


Loading solver: meta-llama/Llama-3.2-1B-Instruct


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Total VRAM (both models): 4.62 GB / 17.1 GB
Headroom                : 12.5 GB
Memory OK


In [9]:
# CELL 9 -- Prompts and generation functions
# CommonsenseQA specific: guide reasons about everyday concepts, not math.
# The guide identifies:
#   1. The core concept being tested
#   2. Relevant real-world knowledge that applies
#   3. Which options match or contradict that knowledge

GUIDE_SYSTEM = (
    "You are a commonsense reasoning assistant for multiple-choice questions.\n"
    "Given a question with options A-E, write 2-3 concrete reasoning steps.\n"
    "Each step must use specific real-world knowledge relevant to the question.\n"
    "Your LAST line must always be: Best answer: <letter> because <one-line reason>\n\n"
    "Rules:\n"
    "- Steps must reference SPECIFIC facts, not vague generalities.\n"
    "- Explicitly eliminate at least one wrong option when possible.\n"
    "- Think about cause-effect, typical human behavior, object properties, or spatial logic.\n"
    "- No markdown. Plain text only.\n\n"
    "BAD example (too vague):\n"
    "  Step 1: Think about the concept in the question.\n"
    "  Step 2: Consider which option makes sense.\n"
    "  Best answer: C because it seems right\n\n"
    "GOOD example (specific reasoning):\n"
    "  Question: Where would you find a penguin in its natural habitat?\n"
    "  Options: A) Amazon rainforest B) Antarctic ice C) Sahara desert D) Rocky mountains E) Coral reef\n"
    "  Step 1: Penguins are native to the Southern Hemisphere and thrive in cold climates.\n"
    "           They live on ice and hunt fish in cold ocean waters.\n"
    "  Step 2: Options A, C, D, E are all warm or tropical -- penguins cannot survive there.\n"
    "           Only option B matches: cold, icy, near ocean.\n"
    "  Best answer: B because penguins are cold-climate birds native to Antarctica\n\n"
    "Apply this pattern to any commonsense question -- cause-effect, location, social norms, "
    "object use, human behavior, or physical properties."
)

SOLVE_SYSTEM = (
    "You are a precise multiple-choice commonsense solver.\n"
    "You are given a question and a reasoning plan.\n"
    "Follow the plan exactly and pick the letter the plan identifies as correct.\n"
    "Do not contradict the plan. Do not reconsider eliminated options.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [letter]\n\n"
    "Example:\n"
    "Plan says: Best answer: B because penguins are cold-climate birds native to Antarctica.\n"
    "The answer is B"
)

SOLVE_BASELINE_SYSTEM = (
    "You are a commonsense question answering assistant.\n"
    "Read the question carefully. Use your general knowledge to pick the best answer.\n"
    "Think step by step if needed.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)

REFINER_SYSTEM = (
    "You are a careful commonsense reasoning checker.\n"
    "You are given a question and a list of candidate answers that are tied.\n"
    "Reason step by step about which answer is most plausible given real-world knowledge.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)


def _generate(model, tokenizer, system_prompt, user_prompt, temperature, max_new_tokens):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def generate_plan(question):
    return _generate(
        guide_model, guide_tok,
        GUIDE_SYSTEM, question,
        CONFIG["guide_temperature"], CONFIG["max_new_tokens"]
    )

def generate_guided(question, plan):
    prompt = f"Plan:\n{plan}\n\nNow answer:\n{question}"
    return _generate(
        resp_model, resp_tok,
        SOLVE_SYSTEM, prompt,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_baseline(question):
    return _generate(
        resp_model, resp_tok,
        SOLVE_BASELINE_SYSTEM, question,
        CONFIG["vote_temperature"], CONFIG["max_new_tokens"]
    )

def generate_refiner(question, tied_answers):
    prompt = (
        f"Question:\n{question}\n\n"
        f"Tied candidate answers: {', '.join(tied_answers)}\n"
        f"Which one is most plausible based on common sense?"
    )
    return _generate(
        resp_model, resp_tok,
        REFINER_SYSTEM, prompt,
        CONFIG["refiner_temperature"], CONFIG["max_new_tokens"]
    )

print("Prompts and generation functions ready")


Prompts and generation functions ready


In [10]:
# CELL 10 -- Voting logic with richer metrics

def vote_and_decide(answers, question, gt_answer=None):
    """
    Majority voting with refiner fallback on ties.
    Filters empty/invalid letter responses before counting.
    Returns dict with all metrics needed for the three angles.
    """
    valid = [a for a in answers if a in VALID_LETTERS]
    if not valid:
        valid = answers  # fallback

    vote_counts  = Counter(valid)
    most_common  = vote_counts.most_common()
    top_answer   = most_common[0][0]
    top_count    = most_common[0][1]
    total        = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        ref_raw  = generate_refiner(question, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_v      = valid + ([ref_ans] if ref_ans in VALID_LETTERS else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted     = total - new_top_c

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "total_votes"      : total,
        "correct_votes"    : correct_votes,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }

print("Voting logic ready")


Voting logic ready


In [11]:
# CELL 11 -- Single question test (verify pipeline end-to-end)

print("=" * 65)
print("SINGLE QUESTION TEST  (CommonsenseQA)")
print("=" * 65)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(f"Question:\n{q}")
print(f"Concept : {item['concept']}")
print(f"GT Answer: {gt}")

# Guided
print("\n[1] Guide generating plan...")
plan = generate_plan(q)
print(f"Plan:\n{plan}")

print(f"\n[2] Guided votes ({CONFIG['n_votes']}x)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw snippet: {raw[:80]}")

g = vote_and_decide(guided_votes, q, gt)
print(f"\n  Result    : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")
print(f"  Vote counts: {g['vote_counts']}")

# Baseline
print("\n[3] Baseline votes (no plan)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw snippet: {raw[:80]}")

b = vote_and_decide(base_votes, q, gt)
print(f"\n  Baseline result : {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print(f"  Vote counts: {b['vote_counts']}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


SINGLE QUESTION TEST  (CommonsenseQA)
Question:
You can do knitting to get the feeling of what?

Options:
A) relaxation
B) arthritis
C) adrenaline
D) your
E) sweater may produced
Concept : knitting
GT Answer: A

[1] Guide generating plan...
Plan:
Step 1: Knitting can be a calming activity that involves repetitive motions, which can help reduce stress and anxiety.
Step 2: Many people engage in knitting as a hobby to unwind and create something useful, like clothing or accessories.
Step 3: While knitting may cause some physical strain on the hands and wrists, it's not typically associated with increasing adrenaline levels.
Best answer: A because knitting is often done for relaxation.

[2] Guided votes (5x)...
  Vote 1: 'A'  |  raw snippet: The answer is A
  Vote 2: 'A'  |  raw snippet: The answer is A
  Vote 3: 'A'  |  raw snippet: The answer is A
  Vote 4: 'A'  |  raw snippet: The answer is A
  Vote 5: 'A'  |  raw snippet: The answer is A

  Result    : A  (GT: A)  CORRECT
  Strategy  :

In [12]:
# CELL 12 -- Full Dual Evaluation Loop
#
# Runs every question TWICE with the SAME questions (seed fixed in Cell 5):
#   Mode A: Guided   (guide plan + solver x5)
#   Mode B: Baseline (solver x5, no plan)
#
# Checkpointing every save_every questions -- safe to interrupt and resume.

print(f"Dual evaluation: {len(test_data)} CommonsenseQA questions")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print(f"Random baseline (chance): 20.0% (1 in 5 options)")
print("-" * 65)

all_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
    print(f"  Guided saved: {len(all_results)}  Baseline saved: {len(base_results)}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="CommonsenseQA Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    # ---- GUIDED -----------------------------------------------
    try:
        plan        = generate_plan(question)
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "concept"          : item.get("concept", ""),
            "gt_answer"        : gt_answer,
            "plan"             : plan,
            "votes"            : g_votes_raw,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "vote_counts"      : g_dec["vote_counts"],
            "total_votes"      : g_dec["total_votes"],
            "correct_votes"    : g_dec["correct_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [GUIDED ERROR idx={idx}]: {e}")
        all_results.append({"mode":"guided","idx":idx,"correct":False,
                             "gt_answer":gt_answer,"final_answer":"",
                             "concept": item.get("concept",""),
                             "strategy":"error","confidence":0.0,
                             "vote_consistency":0.0,"wasted_votes":5,
                             "refiner_used":False,"refiner_correct":None,
                             "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- BASELINE ---------------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "votes"            : b_votes_raw,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "vote_counts"      : b_dec["vote_counts"],
            "total_votes"      : b_dec["total_votes"],
            "correct_votes"    : b_dec["correct_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : b_dec["refiner_correct"],
        })
    except Exception as e:
        print(f"  [BASELINE ERROR idx={idx}]: {e}")
        base_results.append({"mode":"baseline","idx":idx,"correct":False,
                              "gt_answer":gt_answer,"final_answer":"",
                              "strategy":"error","confidence":0.0,
                              "vote_consistency":0.0,"wasted_votes":5,
                              "refiner_used":False,"refiner_correct":None,
                              "vote_counts":{},"total_votes":5,"correct_votes":0})

    # ---- Checkpoint -------------------------------------------
    if (idx + 1) % CONFIG["save_every"] == 0 or (idx + 1) == len(test_data):
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        elapsed = time.time() - t0
        g_acc_so_far = sum(r["correct"] for r in all_results) / len(all_results) * 100
        b_acc_so_far = sum(r["correct"] for r in base_results) / len(base_results) * 100
        print(f"  [{idx+1}/{len(test_data)}] Guided: {g_acc_so_far:.1f}%  "
              f"Baseline: {b_acc_so_far:.1f}%  ({elapsed/60:.1f}min)")

print("\n" + "=" * 65)
print("EVALUATION COMPLETE")
g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
print(f"  Guided   accuracy: {g_acc:.1f}%")
print(f"  Baseline accuracy: {b_acc:.1f}%")
print(f"  Delta            : +{g_acc - b_acc:.1f} pts")
print(f"  Random chance    : 20.0%")
print("=" * 65)
 

Dual evaluation: 900 CommonsenseQA questions
Each question: 5 guided votes + 5 baseline votes
Random baseline (chance): 20.0% (1 in 5 options)
-----------------------------------------------------------------
Starting fresh


CommonsenseQA Eval:   0%|          | 0/900 [00:00<?, ?it/s]

  [25/900] Guided: 68.0%  Baseline: 52.0%  (8.3min)
  [50/900] Guided: 62.0%  Baseline: 56.0%  (16.0min)
  [75/900] Guided: 61.3%  Baseline: 50.7%  (25.7min)
  [100/900] Guided: 61.0%  Baseline: 48.0%  (33.1min)
  [125/900] Guided: 61.6%  Baseline: 48.8%  (41.2min)
  [150/900] Guided: 64.7%  Baseline: 49.3%  (49.6min)
  [175/900] Guided: 64.6%  Baseline: 50.3%  (58.6min)
  [200/900] Guided: 61.0%  Baseline: 49.0%  (68.0min)
  [225/900] Guided: 59.6%  Baseline: 48.9%  (76.7min)
  [250/900] Guided: 59.6%  Baseline: 48.0%  (86.8min)
  [275/900] Guided: 58.9%  Baseline: 46.5%  (96.0min)
  [300/900] Guided: 60.3%  Baseline: 47.3%  (106.0min)
  [325/900] Guided: 60.3%  Baseline: 47.7%  (115.0min)
  [350/900] Guided: 60.0%  Baseline: 46.9%  (124.2min)
  [375/900] Guided: 59.5%  Baseline: 46.9%  (133.2min)
  [400/900] Guided: 59.8%  Baseline: 46.5%  (141.3min)
  [425/900] Guided: 58.8%  Baseline: 45.6%  (150.3min)
  [450/900] Guided: 59.8%  Baseline: 46.4%  (158.9min)
  [475/900] Guided: 59.6%

In [13]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY
# =================================================================
# Baseline : 1.5B solver x5 votes              = 7.5B param-passes
# Guided   : 3B guide x1 + 1.5B solver x5      = 10.5B param-passes
# Upper    : 3B guide x5 votes (ceiling)        = 15.0B param-passes
# CommonsenseQA random chance: 20.0% (5 options A-E)
# =================================================================

G, S, N = CONFIG["guide_params_B"], CONFIG["solver_params_B"], CONFIG["n_votes"]

guided_compute   = (G * 1) + (S * N)
baseline_compute = S * N
upper_compute    = G * N
random_chance    = 20.0

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100

savings_pct = (1 - guided_compute / upper_compute) * 100

g_wasted = sum(r["wasted_votes"] for r in all_results)
b_wasted = sum(r["wasted_votes"] for r in base_results)

ref_triggered = sum(r["refiner_used"] for r in all_results)
ref_correct   = sum(1 for r in all_results if r["refiner_used"] and r.get("refiner_correct"))

strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (CommonsenseQA)")
print("=" * 65)
print(f"  Random chance baseline: {random_chance}% (5 options, A-E)")
print(f"\n  {'Setup':<32} | {'Compute':>10} | {'Accuracy':>9}")
print(f"  {'-'*32}-+-{'-'*10}-+-{'-'*9}")
print(f"  {'Random chance':<32} | {'--':>10} | {random_chance:>8.1f}%")
print(f"  {'Baseline (1.5B x ' + str(N) + ')':<32} | {baseline_compute:>8.1f}B  | {b_acc:>8.1f}%")
print(f"  {'Guided  (3B x1 + 1.5B x' + str(N) + ')':<32} | {guided_compute:>8.1f}B  | {g_acc:>8.1f}%")
print(f"  {'Upper   (3B x ' + str(N) + ')':<32} | {upper_compute:>8.1f}B  | {'(ceiling)':>9}")
print(f"\n  Guided vs baseline gain : +{g_acc - b_acc:.1f} pts")
print(f"  Guided vs random chance : +{g_acc - random_chance:.1f} pts above chance")
print(f"  Compute savings vs upper: {savings_pct:.0f}% cheaper")
print(f"  Wasted votes saved      : {b_wasted - g_wasted}")
if ref_triggered:
    print(f"  Refiner: {ref_triggered} triggered, {ref_correct} correct ({ref_correct/ref_triggered*100:.1f}%)")

print("\n  Strategy breakdown:")
for strat, stats in strategy_stats.items():
    acc = stats["correct"] / stats["n"] * 100 if stats["n"] else 0
    print(f"    {strat:<20}: {stats['n']}q  |  {acc:.1f}% accurate")

a1_data = {
    "dataset": "CommonsenseQA",
    "n_questions": len(all_results),
    "random_chance": random_chance,
    "guided_accuracy": round(g_acc, 1),
    "baseline_accuracy": round(b_acc, 1),
    "accuracy_gain": round(g_acc - b_acc, 1),
    "guided_above_chance": round(g_acc - random_chance, 1),
    "baseline_above_chance": round(b_acc - random_chance, 1),
    "guided_compute_B": guided_compute,
    "baseline_compute_B": baseline_compute,
    "upper_compute_B": upper_compute,
    "compute_savings_pct": round(savings_pct, 1),
    "guided_wasted_votes": g_wasted,
    "baseline_wasted_votes": b_wasted,
    "wasted_votes_saved": b_wasted - g_wasted,
    "refiner_triggered": ref_triggered,
    "refiner_correct": ref_correct,
    "strategy_stats": strategy_stats, 
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(a1_data, f, indent=2)
print(f"\nAngle 1 saved to {CONFIG['angle1_file']}")


ANGLE 1 -- COMPUTE EFFICIENCY  (CommonsenseQA)
  Random chance baseline: 20.0% (5 options, A-E)

  Setup                            |    Compute |  Accuracy
  ---------------------------------+------------+----------
  Random chance                    |         -- |     20.0%
  Baseline (1.5B x 5)              |      7.5B  |     46.6%
  Guided  (3B x1 + 1.5B x5)        |     10.5B  |     58.8%
  Upper   (3B x 5)                 |     15.0B  | (ceiling)

  Guided vs baseline gain : +12.2 pts
  Guided vs random chance : +38.8 pts above chance
  Compute savings vs upper: 30% cheaper
  Wasted votes saved      : 358
  Refiner: 45 triggered, 16 correct (35.6%)

  Strategy breakdown:
    majority            : 855q  |  60.0% accurate
    coin_flip           : 16q  |  31.2% accurate
    refiner_tiebreak    : 29q  |  37.9% accurate

Angle 1 saved to /kaggle/working/csqa_eval/angle1_compute_efficiency.json


In [14]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY
# =================================================================
# Vote consistency = fraction of votes (out of 5) that matched GT.
# Also tracks letter distribution to detect position bias (A-E).
# =================================================================

g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6) 

guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

def bucket(scores):
    return {
        "all_wrong  (0%)":  sum(1 for s in scores if s == 0.0),
        "low       (1-39%)":sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

all_letters_guided   = []
all_letters_baseline = []
for r in all_results:
    all_letters_guided.extend(r["vote_counts"].keys())
for r in base_results:
    all_letters_baseline.extend(r["vote_counts"].keys())
g_letter_dist = Counter(all_letters_guided)
b_letter_dist = Counter(all_letters_baseline)

total_g_letters = sum(g_letter_dist.values())
total_b_letters = sum(b_letter_dist.values())

all_wrong_guided   = sum(1 for r in all_results  if r["vote_consistency"] == 0.0)
all_wrong_baseline = sum(1 for r in base_results if r["vote_consistency"] == 0.0)

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (CommonsenseQA)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio (out of {CONFIG['n_votes']} votes):")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} votes correct avg)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} votes correct avg)")
print(f"    Lift     : {lift:.2f}x")
print(f"\n  Per-question: Guided wins {guided_wins}, Baseline wins {baseline_wins}, Tied {tied}")
print(f"\n  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gn, bn = g_dist[bkt], b_dist[bkt]
    print(f"  {bkt:<22} | {gn:>8} | {bn:>8} | {gn-bn:>+6}")

print(f"\n  Option letter distribution (% of winning votes):")
print(f"  {'Letter':<8} | {'Guided':>8} | {'Baseline':>8}")
print(f"  {'-'*8}-+-{'-'*8}-+-{'-'*8}")
for ltr in sorted(VALID_LETTERS):
    gp = g_letter_dist.get(ltr, 0) / max(total_g_letters, 1) * 100
    bp = b_letter_dist.get(ltr, 0) / max(total_b_letters, 1) * 100
    print(f"  {ltr:<8} | {gp:>7.1f}% | {bp:>7.1f}%")

print(f"\n  All-wrong questions (0 of 5 correct):")
print(f"    Guided: {all_wrong_guided}  |  Baseline: {all_wrong_baseline}  |  Delta: {all_wrong_baseline - all_wrong_guided} fewer with guidance")

a2_data = {
    "dataset": "CommonsenseQA",
    "guided_mean_consistency": round(g_mean, 4),
    "baseline_mean_consistency": round(b_mean, 4),
    "consistency_lift": round(lift, 4),
    "guided_wins": guided_wins,
    "baseline_wins": baseline_wins,
    "tied": tied,
    "guided_distribution": g_dist,
    "baseline_distribution": b_dist,
    "all_wrong_guided": all_wrong_guided,
    "all_wrong_baseline": all_wrong_baseline,
    "guided_letter_dist": {k: round(v/max(total_g_letters,1)*100,1) for k,v in g_letter_dist.items()},
    "baseline_letter_dist": {k: round(v/max(total_b_letters,1)*100,1) for k,v in b_letter_dist.items()},
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(a2_data, f, indent=2)
print(f"\nAngle 2 saved to {CONFIG['angle2_file']}")


ANGLE 2 -- VOTE CONSISTENCY  (CommonsenseQA)

  Mean correct-vote ratio (out of 5 votes):
    Guided   : 56.2%  (2.81 votes correct avg)
    Baseline : 43.6%  (2.18 votes correct avg)
    Lift     : 1.29x

  Per-question: Guided wins 397, Baseline wins 217, Tied 286

  Bucket                 |   Guided | Baseline |   Diff
  -----------------------+----------+----------+-------
  all_wrong  (0%)        |      254 |      282 |    -28
  low       (1-39%)      |       83 |      161 |    -78
  medium  (40-79%)       |      128 |      173 |    -45
  high   (80-100%)       |      435 |      284 |   +151

  Option letter distribution (% of winning votes):
  Letter   |   Guided | Baseline
  ---------+----------+---------
  A        |    22.5% |    35.0%
  B        |    23.9% |    21.1%
  C        |    18.0% |    14.4%
  D        |    23.4% |    15.9%
  E        |    12.1% |    13.4%

  All-wrong questions (0 of 5 correct):
    Guided: 254  |  Baseline: 282  |  Delta: 28 fewer with guidance

Ang

In [15]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION
# =================================================================
# Confidence = fraction of votes that agreed on winning answer.
# ECE = average |accuracy - confidence| weighted by bucket size.
# Lower ECE = better calibrated.
# Random chance is 20% for 5-option MCQ.
# =================================================================

def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),
                           "expected":mid,"gap":round(gap,4)})

    hc     = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    mc     = [r for r in results if 0.60 <= r["confidence"] < 0.80]
    mc_acc = sum(r["correct"] for r in mc) / max(1, len(mc)) * 100
    print(f"  {'ECE':<26}   {ece:.4f}")
    print(f"  High-conf : {len(hc)} questions  |  Accuracy: {hc_acc:.1f}%  |  Confidently WRONG: {false_conf}")
    print(f"  Mid-conf  : {len(mc)} questions  |  Accuracy: {mc_acc:.1f}%")
    return ece, calib_out, false_conf, hc_acc, len(hc), mc_acc, len(mc)

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (CommonsenseQA)")
print("=" * 65)
g_ece, g_calib, g_fc, g_hc_acc, g_hc_n, g_mc_acc, g_mc_n = calibration_report(all_results,  "GUIDED")
b_ece, b_calib, b_fc, b_hc_acc, b_hc_n, b_mc_acc, b_mc_n = calibration_report(base_results, "BASELINE")

ece_improvement = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE improvement : {ece_improvement:.1f}% better calibrated with guidance")
print(f"  ECE threshold   : Guided {'PASSES' if g_ece < 0.10 else 'FAILS'} the <0.10 threshold")
print(f"  False confidence: Guided {g_fc}  vs  Baseline {b_fc}  ({b_fc - g_fc} fewer with guidance)")

a3_data = {
    "dataset": "CommonsenseQA",
    "guided_ece": round(g_ece, 4),
    "baseline_ece": round(b_ece, 4),
    "ece_improvement_pct": round(ece_improvement, 1),
    "guided_calibration": g_calib,
    "baseline_calibration": b_calib,
    "guided_false_confidence": g_fc,
    "baseline_false_confidence": b_fc,
    "guided_high_conf_accuracy": round(g_hc_acc, 1),
    "baseline_high_conf_accuracy": round(b_hc_acc, 1),
    "guided_high_conf_n": g_hc_n,
    "baseline_high_conf_n": b_hc_n,
    "guided_mid_conf_accuracy": round(g_mc_acc, 1),
    "baseline_mid_conf_accuracy": round(b_mc_acc, 1),
    "guided_mid_conf_n": g_mc_n,
    "baseline_mid_conf_n": b_mc_n,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(a3_data, f, indent=2)
print(f"\nAngle 3 saved to {CONFIG['angle3_file']}")


ANGLE 3 -- CONFIDENCE CALIBRATION  (CommonsenseQA)

  [GUIDED]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |   654 |     66.5% |       90% |  0.235 | Poor
  High       (0.60-0.80)     |   187 |     39.6% |       70% |  0.304 | Poor
  Medium     (0.40-0.60)     |    43 |     34.9% |       50% |  0.151 | Poor
  Low        (<0.40)         |    16 |     31.2% |       25% |  0.062 | Good
  ECE                          0.2422
  High-conf : 654 questions  |  Accuracy: 66.5%  |  Confidently WRONG: 219
  Mid-conf  : 187 questions  |  Accuracy: 39.6%

  [BASELINE]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |   523 |     54.3% |       90% |  0.357 | Poor
  High       (0.60-0.80)     |   253 |     37.9% |       70% |  0.321 |

In [16]:
# CELL 16 -- Full Paper Summary (all three angles)

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

n = a1["n_questions"]

print("=" * 68)
print("  COMMONSENSEQA EVALUATION -- PAPER SUMMARY TABLE")
print("=" * 68)
print(f"  Dataset: CommonsenseQA  |  Split: validation  |  N={n}  |  Seed={CONFIG['random_seed']}")
print(f"  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver")
print(f"  Random chance baseline: 20.0%  (5 options, A-E)")
print(f"  NOTE: Guide fine-tuned on math (GSM8K) -- out-of-domain test")
print()

rows = [
    ["Metric",                   "Baseline",                                    "Guided",                                      "Change"],
    ["Overall Accuracy",
     str(a1["baseline_accuracy"]) + "%",
     str(a1["guided_accuracy"]) + "%",
     "+" + str(round(a1["guided_accuracy"]-a1["baseline_accuracy"],1)) + " pts"],
    ["Above Random Chance (20%)",
     "+" + str(a1["baseline_above_chance"]) + " pts",
     "+" + str(a1["guided_above_chance"]) + " pts", ""],
    ["Compute Cost",
     str(a1["baseline_compute_B"]) + "B passes",
     str(a1["guided_compute_B"]) + "B passes",
     str(a1["compute_savings_pct"]) + "% cheaper than ceiling"],
    ["Wasted Votes",
     str(a1["baseline_wasted_votes"]),
     str(a1["guided_wasted_votes"]),
     str(a1["wasted_votes_saved"]) + " fewer"],
    ["Vote Consistency",
     str(round(a2["baseline_mean_consistency"]*100,1)) + "%",
     str(round(a2["guided_mean_consistency"]*100,1)) + "%",
     str(round(a2["consistency_lift"],2)) + "x lift"],
    ["High-Agreement (80-100%)",
     str(a2["baseline_distribution"]["high   (80-100%)"]),
     str(a2["guided_distribution"]["high   (80-100%)"]), ""],
    ["All-Wrong (0/5 correct)",
     str(a2["all_wrong_baseline"]),
     str(a2["all_wrong_guided"]),
     str(a2["all_wrong_baseline"] - a2["all_wrong_guided"]) + " fewer complete failures"],
    ["ECE (lower = better)",
     str(a3["baseline_ece"]),
     str(a3["guided_ece"]),
     str(a3["ece_improvement_pct"]) + "% better"],
    ["High-Conf Accuracy (>=0.80)",
     str(a3["baseline_high_conf_accuracy"]) + "% (n=" + str(a3["baseline_high_conf_n"]) + ")",
     str(a3["guided_high_conf_accuracy"])  + "% (n=" + str(a3["guided_high_conf_n"])  + ")", ""],
    ["False Confidence Count",
     str(a3["baseline_false_confidence"]),
     str(a3["guided_false_confidence"]),
     str(a3["baseline_false_confidence"] - a3["guided_false_confidence"]) + " fewer"],
]

col_w = [32, 26, 26, 30]
header = rows[0]
sep    = "  " + "-+-".join("-" * w for w in col_w)
print("  " + " | ".join(f"{h:<{col_w[i]}}" for i, h in enumerate(header)))
print(sep)
for row in rows[1:]:
    print("  " + " | ".join(f"{str(row[i]):<{col_w[i]}}" for i in range(len(col_w))))

print()
print("=" * 68)
print("  KEY FINDING:")
print(f"  Guided pipeline achieves +{a1['accuracy_gain']:.1f} pts on CommonsenseQA")
print(f"  (out-of-domain: guide trained on math, tested on commonsense).")
print(f"  Calibration improves {a3['ece_improvement_pct']:.1f}% (ECE: {a3['baseline_ece']} -> {a3['guided_ece']}).")
print(f"  This confirms pipeline benefit is NOT math-specific.")
print("=" * 68)


  COMMONSENSEQA EVALUATION -- PAPER SUMMARY TABLE
  Dataset: CommonsenseQA  |  Split: validation  |  N=900  |  Seed=42
  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver
  Random chance baseline: 20.0%  (5 options, A-E)
  NOTE: Guide fine-tuned on math (GSM8K) -- out-of-domain test

  Metric                           | Baseline                   | Guided                     | Change                        
  ---------------------------------+----------------------------+----------------------------+-------------------------------
  Overall Accuracy                 | 46.6%                      | 58.8%                      | +12.2 pts                     
  Above Random Chance (20%)        | +26.6 pts                  | +38.8 pts                  |                               
  Compute Cost                     | 7.5B passes                | 10.5B passes               | 30.0% cheaper than ceiling    
  Wasted Votes                     | 1070                       | 712                